# 🎬 OpenShorts no Google Colab (Aceleração por GPU / CUDA)

Este notebook executa o servidor **OpenShorts** utilizando a GPU do Google Colab (T4 / V100 / A100).

Ao executar, o notebook solicitará interativamente:
1. **GitHub Personal Access Token (PAT)** para clonar o repositório privado `https://github.com/diegofullstackjs/new-open-shorts.git`
2. **Chave de API do Gemini (Google AI Studio)** para a inteligência de cortes e layouts.

---

### 1. Verificar GPU Ativa (CUDA)

In [ ]:
!nvidia-smi

### 2. Autenticação Interativa & Clone do Repositório Privado
Execute a célula abaixo e insira seus tokens nos campos seguros que aparecerão:

In [ ]:
import os
import getpass
from google.colab import userdata

# 1. Obter GitHub Personal Access Token (Tenta Secrets primeiro, senão solicita no prompt seguro)
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not github_token:
    print("🔑 Autenticação GitHub:")
    github_token = getpass.getpass("Digite ou cole seu GitHub Personal Access Token (PAT): ").strip()

# 2. Obter Gemini API Key (Tenta Secrets primeiro, senão solicita no prompt seguro)
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_key = None

if not gemini_key:
    print("\n🤖 Autenticação Gemini AI:")
    gemini_key = getpass.getpass("Digite ou cole sua Gemini API Key (Google AI Studio): ").strip()

# Validar se as chaves foram fornecidas
if not github_token:
    raise ValueError("❌ O GitHub Token é obrigatório para acessar o repositório privado.")
if not gemini_key:
    raise ValueError("❌ A Gemini API Key é obrigatória para processar os vídeos.")

# Configurar variáveis de ambiente do sistema
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['MAX_CONCURRENT_JOBS'] = '2'

# Clonar ou atualizar o repositório privado
repo_url = f"https://{github_token}@github.com/diegofullstackjs/new-open-shorts.git"
target_dir = "/content/new-open-shorts"

if not os.path.exists(target_dir):
    print("\n🚀 Clonando repositório privado new-open-shorts...")
    !git clone {repo_url} {target_dir}
else:
    print("\n🔄 Atualizando repositório existente...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
print("\n✅ Autenticação realizada e repositório carregado com sucesso!")

### 3. Instalar Dependências do Sistema e Pacotes Python

In [ ]:
# Instalar pacotes de sistema necessários (FFmpeg, fontes CJK para legendas, etc.)
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto fonts-noto-cjk libgl1-mesa-glx

# Instalar dependências Python otimizadas para GPU
!pip install -q -r requirements.txt
!pip install -q pycloudflared pyngrok uvicorn

### 4. Iniciar Túnel Público (Cloudflare Tunnel) & Servidor FastAPI
Gera uma URL pública `.trycloudflare.com` para conectar diretamente com a **Extensão Chrome 1-Click Shortify** e automações de canal.

In [ ]:
from pycloudflared import try_cloudflare
import subprocess
import time

# Iniciar o Cloudflare Tunnel na porta 8000
tunnel_url = try_cloudflare(port=8000)
print('='*75)
print(f'🚀 URL PÚBLICA DA API DO OPENSHORTS: {tunnel_url.tunnel}')
print('👉 Cole esta URL nas configurações da sua Extensão Chrome 1-Click Shortify!')
print('='*75)

# Iniciar o servidor FastAPI
!python -m uvicorn app:app --host 0.0.0.0 --port 8000